# Lesson 6.1 — VPC Deep Dive: Subnets, Route Tables, NAT & Security Groups

**Level: Intermediate** | **Exam:** SOA-C03 Domain 5, Task 5.1 (Skill 5.1.1); SAP-C02 Domain 1, Task 1.1

## Learning Objectives
- Design a production VPC with public/private/database subnet tiers
- Configure route tables, NAT Gateways, and Internet Gateways
- Implement security groups and NACLs with the principle of least privilege
- Troubleshoot VPC connectivity using VPC Flow Logs

_Reference: [VPC User Guide](https://docs.aws.amazon.com/vpc/latest/userguide/) | [VPC Flow Logs](https://docs.aws.amazon.com/vpc/latest/userguide/flow-logs.html)_

## VPC Architecture

```
VPC: 10.0.0.0/16 (65,536 IPs)
│
├── AZ-a
│   ├── Public Subnet  10.0.0.0/24    → Route: 0.0.0.0/0 → Internet Gateway
│   ├── Private Subnet 10.0.10.0/24   → Route: 0.0.0.0/0 → NAT Gateway (in public subnet)
│   └── Database Subnet 10.0.20.0/24  → Route: local only (no internet access!)
│
├── AZ-b
│   ├── Public Subnet  10.0.1.0/24
│   ├── Private Subnet 10.0.11.0/24
│   └── Database Subnet 10.0.21.0/24
│
├── Internet Gateway (IGW) → Public subnets connect here
└── NAT Gateway (in each public subnet) → Private subnets connect here
```

## Security Groups vs NACLs

| | Security Groups | NACLs |
|--|----------------|-------|
| **Level** | Instance level | Subnet level |
| **State** | Stateful | Stateless |
| **Rules** | Allow only | Allow + Deny |
| **Applies to** | ENI (network interface) | All traffic entering/leaving subnet |
| **Rule eval** | All rules evaluated | Rules evaluated in order (lowest number first) |

**Stateful vs Stateless:**
- SG stateful: Allow inbound port 80 → return traffic (outbound) automatically allowed
- NACL stateless: Allow inbound port 80 → MUST also explicitly allow outbound ephemeral ports (1024-65535)

## NAT Gateway vs NAT Instance
| | NAT Gateway | NAT Instance |
|--|------------|-------------|
| Managed | Yes (AWS) | No (EC2 instance) |
| Availability | High (99.99% SLA) | Single point of failure |
| Bandwidth | Up to 100 Gbps | Limited by instance type |
| Cost | $0.045/hr + data | EC2 cost (cheaper for low traffic) |
| Security groups | Not applicable | Applicable (filter traffic) |

## CIDR and Subnet Sizing
- /16 = 65,536 IPs (VPC max)
- /24 = 256 IPs (251 usable — AWS reserves 5)
- /28 = 16 IPs (11 usable) — minimum for most subnets
- AWS reserves: network, VPC router, DNS, future use, broadcast

In [ ]:
import boto3, json, ipaddress

ec2 = boto3.client('ec2', region_name='us-east-1')
account_id = boto3.client('sts').get_caller_identity()['Account']
print(f'Account: {account_id}')

In [ ]:
# Analyze existing VPCs and their subnets
vpcs = ec2.describe_vpcs()['Vpcs']

for vpc in vpcs:
    is_default = vpc.get('IsDefault', False)
    name = next((t['Value'] for t in vpc.get('Tags', []) if t['Key'] == 'Name'), 'Unnamed')
    print(f"VPC: {vpc['VpcId']}  {vpc['CidrBlock']:18}  {'[DEFAULT]' if is_default else ''}  {name}")
    
    subnets = ec2.describe_subnets(Filters=[{'Name': 'vpc-id', 'Values': [vpc['VpcId']]}])['Subnets']
    for subnet in sorted(subnets, key=lambda x: x['CidrBlock']):
        subnet_name = next((t['Value'] for t in subnet.get('Tags', []) if t['Key'] == 'Name'), 'Unnamed')
        public = 'PUBLIC' if subnet['MapPublicIpOnLaunch'] else 'PRIVATE'
        print(f"  Subnet: {subnet['SubnetId']}  {subnet['CidrBlock']:18}  {subnet['AvailabilityZone']}  {public}  {subnet_name}")

In [ ]:
# Enable VPC Flow Logs — essential for networking troubleshooting and security analysis

logs = boto3.client('logs', region_name='us-east-1')

# Create a log group for flow logs
LOG_GROUP = '/cloudops-learning/vpc-flow-logs'
try:
    logs.create_log_group(LogGroupName=LOG_GROUP)
    logs.put_retention_policy(LogGroupName=LOG_GROUP, retentionInDays=14)
    print(f'Created log group: {LOG_GROUP}')
except logs.exceptions.ResourceAlreadyExistsException:
    print(f'Log group exists: {LOG_GROUP}')

# Get the default VPC for demonstration
default_vpc = next((v for v in ec2.describe_vpcs()['Vpcs'] if v.get('IsDefault')), None)
if default_vpc:
    vpc_id = default_vpc['VpcId']
    print(f'\nFlow logs will monitor VPC: {vpc_id}')
    print('\nTo enable (requires IAM role for flow logs):')
    print(f'ec2.create_flow_logs(')
    print(f'    ResourceIds=["{vpc_id}"],')
    print(f'    ResourceType="VPC",')
    print(f'    TrafficType="ALL",    # ACCEPT, REJECT, or ALL')
    print(f'    LogDestinationType="cloud-watch-logs",')
    print(f'    LogGroupName="{LOG_GROUP}",')
    print(f'    DeliverLogsPermissionArn="arn:aws:iam::{account_id}:role/FlowLogsRole"')
    print(f')')

print('\nVPC Flow Log fields: version accountid srcaddr dstaddr srcport dstport protocol packets bytes')
print('ACCEPT flow: 10.0.1.5 → 8.8.8.8:443 = outbound HTTPS request')
print('REJECT flow: 0.0.0.0 → 10.0.1.5:22 = blocked SSH attempt from internet')

In [ ]:
# Security Group best practice audit
# Find SGs with risky rules (0.0.0.0/0 on sensitive ports)

risky_ports = [22, 3389, 1433, 3306, 5432, 27017]  # SSH, RDP, MSSQL, MySQL, PostgreSQL, MongoDB

all_sgs = ec2.describe_security_groups()['SecurityGroups']
print(f'Security Group Audit ({len(all_sgs)} groups):\n')

for sg in all_sgs:
    risky_rules = []
    for rule in sg['IpPermissions']:
        from_port = rule.get('FromPort', 0)
        to_port = rule.get('ToPort', 65535)
        
        for ip_range in rule.get('IpRanges', []):
            if ip_range['CidrIp'] in ['0.0.0.0/0']:
                for port in risky_ports:
                    if from_port <= port <= to_port:
                        risky_rules.append(f'Port {port} open to 0.0.0.0/0')
        
        for ipv6_range in rule.get('Ipv6Ranges', []):
            if ipv6_range.get('CidrIpv6') == '::/0':
                for port in risky_ports:
                    if from_port <= port <= to_port:
                        risky_rules.append(f'Port {port} open to ::/0 (IPv6)')
    
    if risky_rules:
        sg_name = sg.get('GroupName', 'Unknown')
        print(f'⚠ RISK: {sg["GroupId"]} ({sg_name})')
        for r in risky_rules:
            print(f'   {r}')

print('\nBest practices:')
print('  SSH (22): Allow only from specific IP ranges or 0.0.0.0/0 only if using SSM Session Manager')
print('  Prefer SSM Session Manager over SSH — no port 22 needed')
print('  RDP (3389): Never expose to 0.0.0.0/0 — use Systems Manager Fleet Manager')

## Exercise

In [ ]:
# TODO: Design a 3-tier VPC architecture and calculate CIDR allocations
# Given VPC CIDR: 10.100.0.0/16
# Regions: us-east-1 (3 AZs: a, b, c)
# Tiers: public, private, database
# 
# Fill in the CIDR blocks for each subnet:
# Public subnets:   need to be /24 (for 251 usable IPs)
# Private subnets:  need to be /24
# Database subnets: can be /27 (27 usable IPs — small DB tier)

architecture = {
    'vpc': '10.100.0.0/16',
    'subnets': {
        'public-a':   {'cidr': 'TODO', 'az': 'us-east-1a', 'type': 'public'},
        'public-b':   {'cidr': 'TODO', 'az': 'us-east-1b', 'type': 'public'},
        'public-c':   {'cidr': 'TODO', 'az': 'us-east-1c', 'type': 'public'},
        'private-a':  {'cidr': 'TODO', 'az': 'us-east-1a', 'type': 'private'},
        'private-b':  {'cidr': 'TODO', 'az': 'us-east-1b', 'type': 'private'},
        'private-c':  {'cidr': 'TODO', 'az': 'us-east-1c', 'type': 'private'},
        'database-a': {'cidr': 'TODO', 'az': 'us-east-1a', 'type': 'database'},
        'database-b': {'cidr': 'TODO', 'az': 'us-east-1b', 'type': 'database'},
        'database-c': {'cidr': 'TODO', 'az': 'us-east-1c', 'type': 'database'},
    }
}

import ipaddress
for name, subnet in architecture['subnets'].items():
    if subnet['cidr'] != 'TODO':
        try:
            net = ipaddress.IPv4Network(subnet['cidr'])
            usable = net.num_addresses - 5  # AWS reserves 5
            print(f'{name:15} {subnet["cidr"]:20} AZ: {subnet["az"]} Usable IPs: {usable}')
        except ValueError as e:
            print(f'{name}: Invalid CIDR — {e}')

## Summary
- **3-tier VPC**: Public (internet-facing) → Private (app tier) → Database (no internet)
- **Security Groups** = stateful, instance-level, allow only. **NACLs** = stateless, subnet-level, allow + deny.
- **NAT Gateway** in public subnet → private subnet instances can reach internet (not the reverse)
- **VPC Flow Logs** = traffic audit trail → send to CloudWatch Logs for Insights queries
- Never expose SSH/RDP to 0.0.0.0/0 — use SSM Session Manager instead

---
**Next:** [Lesson 6.2 — VPC Connectivity: Peering, PrivateLink & Transit Gateway](lesson-6.2-vpc-connectivity.ipynb)